In [8]:
from google.colab import drive
drive.mount('/content/drive')
#uncomment for colab

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%cd drive/MyDrive/Moon\ Mapping/Models/Moon-Mapping/AI\ Models/SwinIR/
%ls
#uncomment for colab

[Errno 2] No such file or directory: 'drive/MyDrive/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR/'
/content/drive/.shortcut-targets-by-id/19TyNbSyd7i1igZMVw4YRX5xonl55yZTb/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR
 architecture.gdoc                                         lightning_logs/
 architecture.txt                                          main_test_swinir.py
 check.png                                                 mismatch.txt
 Checkpoints/                                              models/
 cog.yaml                                                  model_zoo/
 DataSet/                                                 'Moon mapping.pdf'
 DataSet_Gray_8x/                                          network_swinir_state_dict.txt
 DataSet_Gray_High/                                        predict.py
'DataSet_ Grayscale'/                                      README.md
 download-weights.sh                                       testsets/
 experiments/                       

In [10]:
#uncomment for colab
%pip install timm
%pip install torch-lr-finder
%pip install pytorch-msssim
os.environ["CUDA_USE_DSA"] = "1"

In [11]:
from models.network_swinir import default_model_import
import torch
import time
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.optim.lr_scheduler import LambdaLR
from torch_lr_finder import LRFinder
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
import os
from pytorch_msssim import ssim
from models.network_swinir import SwinIR as net

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
model = default_model_import().to(device='cuda')
model.load_state_dict(torch.load('/content/drive/MyDrive/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR/experiments/pretrained_models/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth')['params_ema'])#.to(device='cuda')
print(type(model))
#uncomment for Colab

<class 'models.network_swinir.SwinIR'>


In [13]:
# model.load_state_dict(torch.load(r'G:\My Drive\Moon Mapping\Models\Moon-Mapping\AI Models\SwinIR\experiments\pretrained_models\003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth')['params_ema'])

In [22]:
x = torch.randn(1,3,64,64).to(device = 'cuda')
model.to(device = 'cuda')
x = model(x)
print(x.size())

torch.Size([1, 3, 256, 256])


In [23]:
container = list(model.children())
print(container[0])
x = torch.randn(1,3,64,64).to(device = 'cuda')

Conv2d(3, 240, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


In [32]:
class SwinIR_Modified(nn.Module):
    def __init__(self, device, pretrained = r'/content/drive/MyDrive/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR/experiments/pretrained_models/003_realSR_BSRGAN_DFOWMFC_s64w8_SwinIR-L_x4_GAN.pth'):
        super(SwinIR_Modified, self).__init__()
        model = default_model_import()
        model.load_state_dict(torch.load(pretrained)['params_ema'])
        container = list(model.children())
        del model
        rgb_mean = (0.4488, 0.4371, 0.4040)
        self.mean = torch.Tensor(rgb_mean).view(1, 3, 1, 1).to(device=device)
        self.conv_1 = container[0].to(device=device)
        self.patch_embed = container[1].to(device=device)
        self.patch_unembed = container[2].to(device=device)
        self.drop = container[3].to(device=device)

        # Modify self.RSTB to be a layer
        rstb_layers = container[4]
        self.RSTB = nn.Sequential(*rstb_layers[:-1]).to(device=device)
        self.RSTBf = rstb_layers[-1].to(device=device)

        self.lnorm = container[5].to(device=device)
        self.deconv1 = nn.ConvTranspose2d(in_channels=240, out_channels=3, kernel_size=3, stride=2, padding=1, output_padding=1).to(device=device)
        self.deconv2 = nn.ConvTranspose2d(in_channels=3, out_channels=3, kernel_size=3, stride=2, padding=1, output_padding=1).to(device=device)
        self.to(device=device)
        self.device = device

    def forward(self, x):
        with torch.no_grad():
            self.mean = self.mean.type_as(x)
            x = (x - self.mean) * 1.
            x = self.conv_1(x)
            x_size = x.shape[2], x.shape[3]
            x1 = self.patch_embed(x)
            x1 = self.drop(x1)

            # Call RSTB layers individually
            for rstb in self.RSTB:
                x1 = rstb(x1, x_size)

        x1 = self.RSTBf(x1, x_size)
        x1 = self.lnorm(x1)
        x1 = self.patch_unembed(x1, x_size)
        x1 = self.deconv1(x1)
        x1 = self.deconv2(x1)
        return x1

    def shift(self, device):
        self.mean = self.mean.to(device=device)
        self.conv_1 = self.conv_1.to(device=device)
        self.patch_embed = self.patch_embed.to(device=device)
        self.patch_unembed = self.patch_unembed.to(device=device)
        self.drop = self.drop.to(device=device)

        # Shift RSTB and RSTBf separately
        self.RSTB = self.RSTB.to(device=device)
        self.RSTBf = self.RSTBf.to(device=device)

        self.lnorm = self.lnorm.to(device=device)
        self.deconv1 = self.deconv1.to(device=device)
        self.deconv2 = self.deconv2.to(device=device)
        self.to(device=device)
        self.device = device



In [33]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [34]:
# Instantiate the modified model
model = SwinIR_Modified(device)
model = net(upscale=4, in_chans=1, img_size=64, window_size=8,
                        img_range=1., depths=[6, 6, 6, 6, 6, 6], embed_dim=180, num_heads=[6, 6, 6, 6, 6, 6],
                        mlp_ratio=2, upsampler='nearest+conv', resi_connection='1conv').to(device=device)



In [37]:
# model_m.to(device=device)
# # model_m.train()
# # Print model summary
# from torchsummary import summary
# summary(model, (1, 64, 64))
# x = torch.randn(2,1,64,64).to(device=device)
# x = model(x)
# print(x.size())
# model_m.device
# print(model)


In [38]:

class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_list = [file for file in os.listdir(os.path.join(self.root_dir, 'Train')) if file.endswith('.png')]

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        image = Image.open(os.path.join(self.root_dir, 'Train', img_name))
        gt_image = Image.open(os.path.join(self.root_dir, 'GT', img_name))

        if self.transform:
            image = self.transform(image)
            gt_image = self.transform(gt_image)

        return image, gt_image  # Return tuple instead of dictionary


# Define the dataset and dataloaders
transform = transforms.Compose([
    transforms.ToTensor(),
])

dataset = CustomDataset(root_dir='DataSet_Gray_High', transform=transform)
# Split dataset into train, validation, and test
train_size = len(dataset) #int(0.8 * len(dataset)) + 1
val_size = test_size = 0 #(len(dataset) - train_size) // 2
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=4, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=4, num_workers=4)



/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [39]:
# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1E-06)

In [40]:
class PerceptualLoss(nn.Module):
    def __init__(self, resize=True):
        super(PerceptualLoss, self).__init__()
        # Use pre-trained VGG model (features only)
        blocks = []
        blocks.append(models.vgg16(pretrained=True).features[:4].eval())
        blocks.append(models.vgg16(pretrained=True).features[4:9].eval())
        self.blocks = torch.nn.ModuleList(blocks)

        # Freeze weights
        for block in self.blocks:
            for p in block.parameters():
                p.requires_grad = False

        self.transform = nn.functional.interpolate.Bilinear2d(scale_factor=1.0) if resize else None
        self.register_buffer("mean", torch.tensor([0.485]).view(1, 1, 1, 1))
        self.register_buffer("std", torch.tensor([0.229]).view(1, 1, 1, 1))

    def forward(self, input, target, feature_weights=[1.0, 1.0]):
        """
        Calculates perceptual loss between input and target grayscale images.

        Args:
            input: Tensor of generated images (N, 1, 256, 256).
            target: Tensor of ground truth images (N, 1, 256, 256).
            feature_weights: List of weights for each feature layer (default: [1.0, 1.0]).

        Returns:
            Total perceptual loss (scalar).
        """

        if input.size(1) != 1:
            raise ValueError("Input must be a grayscale image (channel = 1)")
        if target.size(1) != 1:
            raise ValueError("Target must be a grayscale image (channel = 1)")

        # Preprocess input and target
        input = input.repeat(1, 3, 1, 1)  # Replicate grayscale to 3 channels
        target = target.repeat(1, 3, 1, 1)
        input = (input - self.mean) / self.std
        target = (target - self.mean) / self.std

        if self.transform:
            input = self.transform(input)
            target = self.transform(target)

        # Extract features from both images
        f_input = []
        f_target = []
        for block in self.blocks:
            input = block(input)
            target = block(target)
            f_input.append(input)
            f_target.append(target)

        # Calculate loss for each feature layer
        loss = 0.0
        for i, (f_in, f_out) in enumerate(zip(f_input, f_target)):
            loss += feature_weights[i] * torch.nn.functional.l1_loss(f_in, f_out)

        return loss

class CombinedLoss(nn.Module):
  def __init__(self, perceptual_weight=1.0, mse_weight=1.0, resize=False):
    super(CombinedLoss, self).__init__()
    self.perceptual_loss = PerceptualLoss(resize=resize)
    self.mse_loss = nn.MSELoss()
    self.perceptual_weight = perceptual_weight
    self.mse_weight = mse_weight

  def forward(self, input, target):
    # Calculate perceptual and MSE loss
    perceptual_loss = self.perceptual_loss(input, target)
    mse_loss = self.mse_weight * self.mse_loss(input, target)

    # Combine losses with weights
    loss = self.perceptual_weight * perceptual_loss + mse_loss
    return loss


In [41]:
# def criterion(reconstructed, target, alpha=0.8, window_size=11, sigma=1.5, data_range=1):
#     """
#     Computes a combined loss function that includes both SSIM loss and MSE loss (L2 loss).

#     Args:
#         reconstructed (torch.Tensor): Reconstructed image.
#         target (torch.Tensor): Ground truth image.
#         alpha (float): Weight parameter for balancing the two losses (default: 0.5).
#         window_size (int): Size of the Gaussian kernel for SSIM calculation (default: 11).
#         sigma (float): Standard deviation of the Gaussian kernel for SSIM calculation (default: 1.5).
#         data_range (float): The range of the pixel values (typically 1.0 or 255).

#     Returns:
#         torch.Tensor: Combined loss value.
#     """
#     # Calculate SSIM loss
#     ssim_loss = 1 - ssim(reconstructed, target, win_size=window_size, win_sigma=sigma, data_range=data_range)

#     # Calculate MSE loss
#     mse_loss = F.mse_loss(reconstructed, target)

#     # Combine the two losses using a weighted sum
#     combined_loss = alpha * ssim_loss + (1 - alpha) * mse_loss

#     return combined_loss

criterion = CombinedLoss().to(device=device)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:03<00:00, 163MB/s]


In [43]:
checkpoint = torch.load(r'/content/drive/MyDrive/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR/Checkpoints/PerceptualLoss/checkpoint_epoch_59.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
loss = checkpoint['loss']
print(epoch, loss, optimizer.param_groups[0]['lr'])

58 tensor(1.7271, device='cuda:0', requires_grad=True) 1.1468800000000004e-08


In [44]:
ckpt = torch.load(r'/content/drive/MyDrive/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR/Checkpoints/SSIM Loss - Pretrained Chkpts/checkpoint_epoch_33.pt')
print(ckpt['loss'])
print(ckpt['model_state_dict'])

Streaming output truncated to the last 5000 lines.
        -2.9635e-01,  1.9154e-02, -1.5586e-01, -7.3583e-02, -2.2644e-02,
        -7.4969e-02,  4.7523e-02,  1.5043e-01, -9.4398e-02, -2.2232e-01,
         1.8499e-01,  1.4641e-01,  9.4129e-02, -2.3082e-02,  1.6283e-01,
        -1.6035e-02,  9.1393e-02,  1.5354e-01,  6.3634e-02, -1.1630e-02,
         8.1797e-02, -9.7152e-02, -3.8222e-02,  1.9865e-03,  1.5631e-01,
         3.5563e-02, -5.8146e-02,  9.1854e-02,  2.7642e-02,  4.3422e-02,
        -4.7860e-02,  2.1173e-01,  6.5725e-02,  5.6068e-02, -1.1554e-02,
        -1.9970e-01,  5.7519e-02, -1.7318e-01, -1.0632e-02,  5.0473e-02,
         1.2957e-01, -4.6382e-02, -3.6456e-02,  4.8563e-02,  2.2143e-02,
        -1.6353e-01, -1.4895e-01, -1.9451e-01,  9.3365e-02, -9.1570e-02,
        -1.2650e-01,  4.5340e-02, -1.4834e-02,  1.3259e-01, -1.1207e-01,
        -2.1313e-02, -9.1804e-03,  6.1896e-02, -1.1181e-02,  6.1836e-02,
         1.4761e-01, -8.1542e-02, -5.4612e-02,  6.3644e-03, -1.1445e-01,


In [48]:
optimal_lr = 7e-6
for param_group in optimizer.param_groups:
  param_group['lr'] = optimal_lr

In [53]:
# Create learning rate finder
# lr_finder = LRFinder(model, optimizer, criterion, device=device)
# lr_finder.range_test(train_loader, start_lr=1e-7, end_lr=1, num_iter=100)  # Adjust end_lr and num_iter as needed

# # Plot the learning rate range test
# ax, lr = lr_finder.plot()

# # Get recommended learning rate
# # recommended_lr = lr_finder.history['lr'][lr_finder.history['loss'].index(lr_finder.best_loss)]
# print(f"Recommended learning rate: {lr}")

# # Reset model and optimizer
# # model = SwinIR_Modified(device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
# # optimizer = torch.optim.Adam(model.parameters(), lr=recommended_lr)

# # Optionally, reset LR scheduler if you use one
# # scheduler = LambdaLR(optimizer, lr_lambda=lambda epoch: 1)  # Modify as needed

# # Now you can proceed with your training loop using the recommended learning rate


In [ ]:
start_epoch = 59
num_epochs = 100
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.4, patience=3, threshold=1e-2, verbose=True)

for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss = 0.0
    train_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}', leave=True)
    for i, data in enumerate(train_bar, 0):
        inputs, labels = data[0].to(device=device), data[1].to(device=device)
        # start_time = time.perf_counter()
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        # end_time = time.perf_counter()
        # print(end_time-start_time)
        train_bar.set_postfix(train_loss=f"{(running_loss / (i + 1)):.5f}")

    # Save checkpoint
    checkpoint_path = rf"Checkpoints/checkpoint_epoch_{epoch + 1}.pt"
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, checkpoint_path)

    # Validation loop
    # model.eval()
    # val_loss = 0.0
    # val_bar = tqdm(val_loader, desc=f'Validation', leave=True)
    # with torch.no_grad():
    #     for i, data in enumerate(val_bar, 0):
    #         inputs, labels = data[0].to(device=device), data[1].to(device=device)
    #         outputs = model(inputs)
    #         val_loss += criterion(outputs, labels).item()

    #         val_bar.set_postfix(val_loss=val_loss / (i + 1))
    scheduler.step(running_loss/len(train_bar))
    print(scheduler.get_last_lr())
    # # Adjust learning rate using LRFinder
    # lr_finder = LRFinder(model, optimizer, criterion, device=device)
    # lr_finder.range_test(train_loader=train_loader, start_lr=1e-7, end_lr=1e-2, num_iter=100)
    # ax, optimal_lr = lr_finder.plot()  # Plot the LR range test graph
    # lr_finder.reset()  # Reset the model and optimizer to their initial state

    # Update optimizer with the optimal learning rate
    # for param_group in optimizer.param_groups:
    #     param_group['lr'] = optimal_lr


Epoch 60/100: 100%|██████████| 3123/3123 [43:17<00:00,  1.20it/s, train_loss=25.65101]


[7e-06]


Epoch 61/100: 100%|██████████| 3123/3123 [42:26<00:00,  1.23it/s, train_loss=6.03592]


[7e-06]


Epoch 62/100: 100%|██████████| 3123/3123 [42:24<00:00,  1.23it/s, train_loss=2.92675]


[7e-06]


Epoch 63/100: 100%|██████████| 3123/3123 [42:30<00:00,  1.22it/s, train_loss=2.13766]


[7e-06]


Epoch 64/100:  11%|█         | 328/3123 [04:28<38:05,  1.22it/s, train_loss=2.00879]

In [1]:
img = Image.open(r'DataSet_Gray_High/Train/23.png')
img.show()
img = transforms.PILToTensor()(img).to(device=device)
img = img.unsqueeze(0).to(dtype=torch.float32)
model.eval()
img = model(img).to(device='cpu')
img = img.squeeze(0)
print(img.size())
img = transforms.ToPILImage()(img).save('check.png')

NameError: name 'Image' is not defined

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import WandbLogger
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class model1(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = net(upscale=4, in_chans=1, img_size=64, window_size=8,
                        img_range=1., depths=[6, 6, 6, 6, 6, 6], embed_dim=180, num_heads=[6, 6, 6, 6, 6, 6],
                        mlp_ratio=2, upsampler='nearest+conv', resi_connection='1conv')
        self.criterion = CombinedLoss()

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=1e-4)
        return optimizer

    def training_step(self, batch, batch_idx):
        # access your optimizers with use_pl_optimizer=False. Default is True
        inputs, labels = batch[0], batch[1]
        outputs = self.model(inputs)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss)
        return loss

checkpoint_callback = ModelCheckpoint(
    dirpath='Checkpoints/model_1_pl_perceptual_loss',
    filename='{epoch}-{train_loss:.2f}',
    save_top_k=1,  # Save only the best model based on training loss
    monitor='train_loss',  # Metric to monitor for improvement
    mode='min',  # Save the model with the training validation loss
    every_n_epochs=1  # Save a checkpoint after every epoch
)


In [ ]:
ckpt_path = "Checkpoints/model_1_pl_perceptual_loss/epoch=26-train_loss=1.03.ckpt"
model = model1.load_from_checkpoint(ckpt_path)
wandb_logger = WandbLogger(
    save_dir='Checkpoints/model_1_pl_perceptual_loss/wandb',
    project="Moon_mapping",
    name="model1_run",
)
torch.set_float32_matmul_precision('high')
Trainer = pl.Trainer(accelerator="gpu", devices=1, logger=wandb_logger, callbacks=checkpoint_callback)
Trainer.fit(model, train_loader, ckpt_path=ckpt_path)
wandb_logger.finalize(status="finished")

/home/sukhvansh/.local/lib/python3.8/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/sukhvansh/.local/lib/python3.8/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/sukhvansh/.local/lib/python3.8/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
GPU available: True (cuda), used: True

/home/sukhvansh/.local/lib/python3.8/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:653: Checkpoint directory /mnt/g/My Drive/Moon Mapping/Models/Moon-Mapping/AI Models/SwinIR/Checkpoints/model_1_pl_perceptual_loss exists and is not empty.
Restoring states from the checkpoint path at Checkpoints/model_1_pl_perceptual_loss/epoch=26-train_loss=1.03.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type         | Params
-------------------------------------------
0 | model     | SwinIR       | 11.7 M
1 | criterion | CombinedLoss | 260 K 
-------------------------------------------
11.7 M    Trainable params
260 K     Non-trainable params
12.0 M    Total params
47.885    Total estimated model params size (MB)
Restored all states from the checkpoint at Checkpoints/model_1_pl_perceptual_loss/epoch=26-train_loss=1.03.ckpt


Epoch 42:  40%|████      | 1250/3123 [10:19<15:27,  2.02it/s, v_num=o2ws]